# Arabic NER — Benchmark Evaluation & Extraction

Worked example of the three-stage NER evaluation workflow applied to Arabic.

**Benchmark dataset:** WikiANN (`wikiann`, `ar`) — public, available on HuggingFace  
**Model:** `hatmimoha/arabic-ner` — Arabic NER model, HuggingFace  
**Extraction:** dummy Arabic sentences demonstrating the `NamedEntityExtractions` pipeline output

See [WORKFLOW.md](../../WORKFLOW.md) for full methodology documentation.

## Setup

**Google Colab:** run the install cell below.  
**Local:** install with `pip install git+https://github.com/ay94/multilingual-ner.git`

In [ ]:
%%capture
!pip install git+https://github.com/ay94/multilingual-ner.git transformers datasets seqeval sentencepiece

In [ ]:
import pandas as pd
import json
from multilingual_ner.evaluation import ReadNERData, ModelEvaluation, align_dataset, check_labels
from multilingual_ner.extraction import NamedEntityExtractions

---
## Stage 1 — Model & Benchmark Selection

**Model selected:** `hatmimoha/arabic-ner`  
- License: Apache 2.0  
- Trained on Arabic NER data  
- Available on HuggingFace — no special preprocessing required

**Benchmark selected:** WikiANN Arabic (`wikiann`, `ar`)  
- Public multilingual NER dataset  
- Labels: PER, LOC, ORG (BIO format)  
- Not used in hatmimoha training data

**Annotation scheme check:** WikiANN uses `B-PER / I-PER / B-LOC / I-LOC / B-ORG / I-ORG / O`.  
The model's label scheme is verified in Stage 2 before defining the alignment.

---
## Stage 2 — Benchmark Evaluation

In [ ]:
# WikiANN label map (integer IDs used in the HuggingFace dataset)
label_map = {
    'O':     0,
    'B-PER': 1,
    'I-PER': 2,
    'B-ORG': 3,
    'I-ORG': 4,
    'B-LOC': 5,
    'I-LOC': 6,
}

reader = ReadNERData()
words, labels = reader.read_dataset('wikiann', label_map, lang='ar')

print(f'Sentences loaded: {len(words)}')
print('Label set:', check_labels(labels))

In [ ]:
# Load model and inspect label scheme before defining alignment
model_eval = ModelEvaluation('hatmimoha/arabic-ner')
print('Model labels:', model_eval.model.config.id2label)

In [ ]:
# Align model output labels to standard scheme (PER, LOC, ORG)
# Adjust if model.config.id2label shows different label names
alignment = {
    'B-PERS': 'B-PER',
    'I-PERS': 'I-PER',
    'B-LOC':  'B-LOC',
    'I-LOC':  'I-LOC',
    'B-ORG':  'B-ORG',
    'I-ORG':  'I-ORG',
    'B-MISC': 'O',
    'I-MISC': 'O',
    'O':      'O',
}

In [ ]:
# Run benchmark evaluation
model_eval = ModelEvaluation('hatmimoha/arabic-ner', alignment)
evaluation_output = model_eval.evaluate_model(words, labels)

In [ ]:
# Entity-level results (seqeval) — F1 by entity type
seqeval_results = evaluation_output.get_classification('Seqeval')
seqeval_results

In [ ]:
# Token-level results (sklearn) — boundary-sensitive flat F1
sklearn_results = evaluation_output.get_classification('Sklearn')
sklearn_results

---
## Stage 3 — Extraction on Project Data

Demonstrates the `NamedEntityExtractions` pipeline on dummy Arabic sentences.  
In a real project, replace `sample_data` with the actual project DataFrame.

Required columns: `text`, `message_id`, `accountId`  
(`message_id` and `accountId` are project-specific identifiers — replaced with dummy values here)

In [ ]:
# Dummy Arabic sentences — replace with real project data
sample_data = pd.DataFrame({
    'text': [
        'زار الرئيس محمد مرسي القاهرة الأسبوع الماضي.',
        'أعلنت منظمة الصحة العالمية عن اكتشاف جديد في جنيف.',
        'التقى وزير الخارجية عمرو موسى بنظيره الأمريكي في نيويورك.',
        'أسست شركة مايكروسوفت في ألبوكيرك بولاية نيومكسيكو عام 1975.',
        'فازت مصر على المغرب في نهائي كأس أمم أفريقيا.',
    ],
    'message_id': ['msg_001', 'msg_002', 'msg_003', 'msg_004', 'msg_005'],
    'accountId':  ['acc_1',   'acc_1',   'acc_2',   'acc_2',   'acc_3'],
})

sample_data

In [ ]:
# Run extraction
extractor = NamedEntityExtractions(
    model_name='hatmimoha/arabic-ner',
    project_data=sample_data,
    text_col='text',
    batch_size=4,
)

json_schema, output_df, post_processed, raw_outputs = extractor.extract_outputs()

In [ ]:
# Structured output — entity lists by type per sentence
output_df[['text', 'PER', 'LOC', 'ORG', 'MISC']]

In [ ]:
# Full JSON record for first sentence
print(json.dumps(json_schema[0], ensure_ascii=False, indent=2))

In [ ]:
# Raw post-processed extractions — (entity, type, start, end) tuples
for i, (sentence, entities) in enumerate(zip(sample_data['text'], post_processed)):
    print(f'\n[{i+1}] {sentence}')
    for entity, etype, start, end in entities:
        print(f'    {etype:6}  {entity}  (chars {start}-{end})')